# TikTok Day-30 View Prediction — v2

**What changed vs the first notebook, and why:**

| # | Change | Reason |
|---|--------|--------|
| 1 | Creator historical-performance features, computed **fresh inside every CV fold** (leave-one-out, log-scale, smoothed) | The account-size features (followers, video count) say nothing about how well *this creator's videos* actually tend to perform. A first attempt at this leaked/overfit because it aggregated the raw (power-law) target with weak smoothing and a fixed tree count — fixed here. |
| 2 | LightGBM trained with a **Tweedie objective** directly on raw views, alongside the existing log-target model | Raw RMSE is the competition metric, and it's dominated by a handful of viral videos. A log-target model is pulled toward the bulk of the data and is *conservative exactly where the metric is most sensitive* (confirmed in the v1 error analysis: 9 of the top-10 errors were under-predictions). Tweedie is built for skewed count data and optimizes closer to the metric that matters. |
| 3 | **Regularized stacking** instead of pure equal-weight blending | Equal weights avoid overfitting to the few viral videos but leave performance on the table. A ridge-shrunk blend (learned weights pulled toward equal via an L2 penalty, swept over a shrinkage grid and picked by CV) gets some of the benefit of learned weights without the overfitting risk. |
| 4 | Kept: engagement/creator/metadata feature engineering, log & growth targets, repeated 5-fold CV, post-hoc clipping | These were already validated as helpful in the first notebook (clipping alone cut LightGBM's raw RMSE by ~45%). |

This notebook is built from the diagnostics of the first one but has **not been executed against your data** — please run it top to bottom and share the printed numbers so we can iterate.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

train = pd.read_csv('train_videos.csv')
test = pd.read_csv('test_videos.csv')
engagement = pd.read_csv('engagement_daily.csv')
creators = pd.read_csv('creators_daily.csv')
sample_submission = pd.read_csv('sample_submission.csv')

for df in [train, test]:
    for c in ['ratio', 'desc_language', 'music_selected_from', 'topic']:
        df[c] = df[c].fillna('unknown')

print('train:', train.shape, '| test:', test.shape)

## 2. Feature Engineering

Unchanged from the first notebook: engagement features use only Days 1-5 (Day 0 is missing for
~60% of videos, see v1's data-quality check), plus creator account-size features and video
metadata. The creator's **historical performance** (new in this notebook) is deliberately *not*
built here — it depends on which rows are "training" in a given fold, so it is computed later,
inside the CV loop and again at the very end for the final fit.

In [ ]:
ENG_METRICS = ['play_count', 'like_count', 'comment_count', 'share_count',
               'collect_count', 'download_count', 'whatsapp_share_count']

def engagement_features(eng, video_ids):
    eng = eng[eng['video_id'].isin(video_ids)]
    out = pd.DataFrame(index=pd.Index(video_ids, name='video_id'))
    tables = {m: eng.pivot(index='video_id', columns='days_since_post', values=m)
                    .reindex(index=out.index, columns=range(6)) for m in ENG_METRICS}
    play = tables['play_count']

    out['n_days_observed'] = play[[1, 2, 3, 4, 5]].notna().sum(axis=1)
    out['has_day0'] = play[0].notna().astype(int)
    out['play_day0'] = play[0]

    last = {}
    for m in ENG_METRICS:
        t = tables[m][[1, 2, 3, 4, 5]].ffill(axis=1).bfill(axis=1)
        last[m] = t[5]
        out[f'{m}_last'] = t[5]
        out[f'{m}_d1'] = t[1]
        out[f'{m}_d3'] = t[3]

    f = play[[1, 2, 3, 4, 5]].ffill(axis=1).bfill(axis=1)
    for d in [2, 3, 4, 5]:
        out[f'play_inc_d{d}'] = f[d] - f[d - 1]
    out['play_growth_1_5'] = f[5] / f[1].replace(0, np.nan)
    out['play_growth_4_5'] = f[5] / f[4].replace(0, np.nan)
    out['play_growth_1_3'] = f[3] / f[1].replace(0, np.nan)
    out['inc_ratio_5_2'] = out['play_inc_d5'] / out['play_inc_d2'].replace(0, np.nan)
    out['inc_ratio_5_3'] = out['play_inc_d5'] / out['play_inc_d3'].replace(0, np.nan)
    out['inc_ratio_5_4'] = out['play_inc_d5'] / out['play_inc_d4'].replace(0, np.nan)   # new: fine-grained deceleration

    views = last['play_count'].replace(0, np.nan)
    for m, name in [('like_count', 'like'), ('comment_count', 'comment'), ('share_count', 'share'),
                    ('collect_count', 'collect'), ('download_count', 'download'),
                    ('whatsapp_share_count', 'whatsapp')]:
        out[f'{name}_to_view'] = last[m] / views
    out['like_to_view_d1'] = out['like_count_d1'] / out['play_count_d1'].replace(0, np.nan)
    out['day1_share_of_last'] = out['play_count_d1'] / views

    return out.replace([np.inf, -np.inf], np.nan)


def creator_features(videos, creators_df):
    creators_df = creators_df.copy()
    creators_df['date'] = pd.to_datetime(creators_df['date'])
    v = videos[['video_id', 'author_id', 'create_date']].copy()
    v['create_date'] = pd.to_datetime(v['create_date'])
    merged = pd.merge_asof(
        v.sort_values('create_date'),
        creators_df.sort_values('date')[['author_id', 'date', 'follower_count', 'following_count',
                                          'total_favorited', 'video_count']],
        left_on='create_date', right_on='date', by='author_id', direction='backward'
    )
    merged = merged.rename(columns={'follower_count': 'creator_followers', 'following_count': 'creator_following',
                                     'total_favorited': 'creator_total_favorited', 'video_count': 'creator_video_count'})
    return merged[['video_id', 'creator_followers', 'creator_following', 'creator_total_favorited', 'creator_video_count']]


def video_meta_features(df):
    df = df.copy()
    df['create_time'] = pd.to_datetime(df['create_time'])
    df['post_hour'] = df['create_time'].dt.hour
    df['post_dayofweek'] = df['create_time'].dt.dayofweek
    df['post_month'] = df['create_time'].dt.month
    df['is_weekend'] = df['post_dayofweek'].isin([5, 6]).astype(int)
    df['text_missing'] = df['word_count'].isna().astype(int)
    keep = ['video_id', 'author_id', 'duration', 'ratio', 'desc_language', 'is_english', 'created_by_ai', 'is_ads',
            'music_selected_from', 'word_count', 'emoji_count', 'question_count', 'hashtag_count', 'speaking_rate',
            'topic', 'anger', 'joy', 'surprise', 'sadness', 'disgust', 'fear', 'post_hour', 'post_dayofweek',
            'post_month', 'is_weekend', 'text_missing']
    return df[keep]


def build_dataset(videos, eng, creators_df):
    meta = video_meta_features(videos)
    eng_f = engagement_features(eng, videos['video_id'].values).reset_index()
    cre_f = creator_features(videos, creators_df)
    out = meta.merge(eng_f, on='video_id', how='left').merge(cre_f, on='video_id', how='left')
    out['log_views_over_followers'] = np.log1p(out['play_count_last']) - np.log1p(out['creator_followers'])
    return out

train_feat = build_dataset(train, engagement, creators)
test_feat = build_dataset(test, engagement, creators)
train_feat['target_day30_views'] = train['target_day30_views'].values
print(train_feat.shape, test_feat.shape)

## 3. Target, Validation Strategy & Creator Historical-Performance Encoding

Two target formulations as in v1 (`log`, `growth`), plus a third: **`tweedie`**, which trains
directly on raw `target_day30_views` with a Tweedie loss — no log transform, so it stays close
to the raw-RMSE metric the leaderboard actually scores.

The creator encoding functions below take an explicit set of "allowed" training positions and
must be called **inside** every fold — never on the full dataset up front — or the model sees a
leaked version of its own target through the creator's average.

In [ ]:
from sklearn.model_selection import KFold

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

y_raw = train_feat['target_day30_views'].values.astype('float64')
y_log = np.log1p(y_raw)
LOG_Y = y_log  # alias used by the encoder below

last_train = train_feat['play_count_last'].fillna(0).values
last_test = test_feat['play_count_last'].fillna(0).values
base_train, base_test = np.log1p(last_train), np.log1p(last_test)

X = train_feat.drop(columns=['target_day30_views', 'video_id', 'author_id'])
X_test = test_feat.drop(columns=['video_id', 'author_id'])
assert list(X.columns) == list(X_test.columns)

cat_cols = X.select_dtypes(exclude='number').columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
for c in cat_cols:
    cats = sorted(set(X[c].astype(str)) | set(X_test[c].astype(str)))
    X[c] = pd.Categorical(X[c].astype(str), categories=cats)
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=cats)

def y_target(mode):
    if mode == 'log':
        return y_log
    if mode == 'growth':
        return y_log - base_train
    if mode == 'tweedie':
        return y_raw
    raise ValueError(mode)

def to_views(mode, pred, base):
    if mode == 'log':
        return np.expm1(pred)
    if mode == 'growth':
        return np.expm1(base + pred)
    if mode == 'tweedie':
        return np.clip(pred, 0, None)
    raise ValueError(mode)

def finalize(views, last):
    return np.maximum(np.clip(views, 0, None), last)

top10 = np.argsort(-y_raw)[:10]
mask_no_top10 = np.ones(len(y_raw), dtype=bool)
mask_no_top10[top10] = False

def score(views):
    return {'raw': rmse(y_raw, views), 'log': rmse(y_log, np.log1p(views)),
            'no_top10': rmse(y_raw[mask_no_top10], views[mask_no_top10])}

N_SEEDS = 2
splits = [list(KFold(5, shuffle=True, random_state=s).split(X)) for s in range(N_SEEDS)]
hold_tr, hold_va = splits[0][0]
print('categorical columns:', cat_cols)
print('hold-out:', len(hold_tr), 'train rows /', len(hold_va), 'validation rows')

In [ ]:
# ---- Creator historical performance: leak-free out-of-fold target encoding -------------------
AUTHOR_TRAIN = train_feat['author_id'].values
AUTHOR_TEST = test_feat['author_id'].values
GLOBAL_MEAN_LOGY = float(LOG_Y.mean())
SMOOTH_K = 60.0   # most creators have 0-2 other videos per fold; lean hard on the global mean

def author_group_stats(train_positions):
    """sum / count / median of log1p(target) for each author, using ONLY the given train rows."""
    df = pd.DataFrame({'author': AUTHOR_TRAIN[train_positions], 'y': LOG_Y[train_positions]})
    return df.groupby('author')['y'].agg(['sum', 'count', 'median'])

def encode_author_features(author_arr, stats, own_y=None):
    """
    Map author-level `stats` onto `author_arr`.
    own_y given  -> these rows ARE the rows `stats` was built from: exact leave-one-out mean.
    own_y None   -> these rows are held-out/new: use the full training aggregate directly.
    """
    s = pd.DataFrame({'author': author_arr}).merge(stats, left_on='author', right_index=True, how='left')
    n = s['count'].fillna(0).values
    total = s['sum'].fillna(GLOBAL_MEAN_LOGY).values
    if own_y is not None:
        loo_n = np.maximum(n - 1, 0)
        raw_mean = np.where(loo_n > 0, (total - own_y) / np.maximum(loo_n, 1), GLOBAL_MEAN_LOGY)
        n_used = loo_n
    else:
        raw_mean = np.where(n > 0, total / np.maximum(n, 1), GLOBAL_MEAN_LOGY)
        n_used = n
    mean_smooth = (raw_mean * n_used + GLOBAL_MEAN_LOGY * SMOOTH_K) / (n_used + SMOOTH_K)
    out = pd.DataFrame(index=range(len(author_arr)))
    out['author_mean_log_target'] = mean_smooth
    out['author_median_log_target'] = s['median'].fillna(GLOBAL_MEAN_LOGY).values
    out['author_n_prior_videos'] = n_used
    return out

def add_encoding(Xdf, author_arr, train_positions, own_y=None):
    stats = author_group_stats(train_positions)
    enc = encode_author_features(author_arr, stats, own_y=own_y)
    return pd.concat([Xdf.reset_index(drop=True), enc], axis=1)

print('Encoder ready. New columns per call: author_mean_log_target, author_median_log_target, author_n_prior_videos')

## 4. Models

Four families as in v1 (LightGBM, Random Forest, Ridge, CatBoost), plus LightGBM's Tweedie
variant. Hyperparameters are picked once on the 80/20 hold-out (with the encoding applied to that
split only, since it is fixed) and then reused inside the CV loop.

In [ ]:
import lightgbm as lgb

def lgb_params(lr, objective='regression', seed=SEED):
    p = dict(objective=objective, metric='rmse', learning_rate=lr, num_leaves=31, min_data_in_leaf=20,
             feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, seed=seed, verbosity=-1)
    if objective == 'tweedie':
        p['tweedie_variance_power'] = 1.3
    return p

# encoding for the fixed hold-out split, used only for hyperparameter search
hold_stats = author_group_stats(hold_tr)
X_hold_tr = pd.concat([X.iloc[hold_tr].reset_index(drop=True),
                        encode_author_features(AUTHOR_TRAIN[hold_tr], hold_stats, own_y=LOG_Y[hold_tr])], axis=1)
X_hold_va = pd.concat([X.iloc[hold_va].reset_index(drop=True),
                        encode_author_features(AUTHOR_TRAIN[hold_va], hold_stats, own_y=None)], axis=1)

LGB_CFG, lr_rows = {}, []
lgb_modes = [('log', 'regression'), ('growth', 'regression'), ('tweedie', 'tweedie')]
for mode, obj in lgb_modes:
    y_mode = y_target(mode)
    for lr in tqdm([0.03, 0.05, 0.08], desc=f'LightGBM lr check ({mode})'):
        dtr = lgb.Dataset(X_hold_tr, label=y_mode[hold_tr], categorical_feature=cat_cols)
        dva = lgb.Dataset(X_hold_va, label=y_mode[hold_va], reference=dtr, categorical_feature=cat_cols)
        booster = lgb.train(lgb_params(lr, obj), dtr, num_boost_round=2000, valid_sets=[dva], valid_names=['val'],
                             callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
        lr_rows.append((mode, lr, booster.best_iteration, booster.best_score['val']['rmse']))

lr_results_df = pd.DataFrame(lr_rows, columns=['target', 'learning_rate', 'best_iteration', 'val_rmse'])
display(lr_results_df)
for mode, obj in lgb_modes:
    best = lr_results_df[lr_results_df['target'] == mode].sort_values('val_rmse').iloc[0]
    LGB_CFG[mode] = dict(lr=float(best['learning_rate']), iters=max(int(best['best_iteration']), 1), objective=obj)
print(LGB_CFG)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X_rf, X_test_rf = X.copy(), X_test.copy()
for c in cat_cols:
    X_rf[c] = X_rf[c].cat.codes
    X_test_rf[c] = X_test_rf[c].cat.codes
X_rf, X_test_rf = X_rf.fillna(-1), X_test_rf.fillna(-1)

RF_TREES = 200
X_hold_tr_rf = pd.concat([X_rf.iloc[hold_tr].reset_index(drop=True),
                           encode_author_features(AUTHOR_TRAIN[hold_tr], hold_stats, own_y=LOG_Y[hold_tr])], axis=1)
X_hold_va_rf = pd.concat([X_rf.iloc[hold_va].reset_index(drop=True),
                           encode_author_features(AUTHOR_TRAIN[hold_va], hold_stats, own_y=None)], axis=1)
rf_check = RandomForestRegressor(n_estimators=RF_TREES, max_features=0.33, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
rf_check.fit(X_hold_tr_rf, y_target('log')[hold_tr])
print('RF hold-out log RMSE:', rmse(y_target('log')[hold_va], rf_check.predict(X_hold_va_rf)))

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

def signed_log1p(a):
    return np.sign(a) * np.log1p(np.abs(a))

ridge_num_cols = num_cols + ['author_mean_log_target', 'author_median_log_target', 'author_n_prior_videos']

def make_ridge_prep():
    return ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                           ('log', FunctionTransformer(signed_log1p)),
                           ('scale', StandardScaler())]), ridge_num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

alpha_rows = []
for alpha in tqdm([1, 3, 10, 30, 100], desc='Ridge alpha check'):
    pipe = Pipeline([('prep', make_ridge_prep()), ('model', Ridge(alpha=alpha))])
    pipe.fit(X_hold_tr, y_target('log')[hold_tr])
    alpha_rows.append((alpha, rmse(y_target('log')[hold_va], pipe.predict(X_hold_va))))
alpha_df = pd.DataFrame(alpha_rows, columns=['alpha', 'val_rmse_log'])
display(alpha_df)
RIDGE_ALPHA = float(alpha_df.sort_values('val_rmse_log').iloc[0]['alpha'])
print('Best alpha:', RIDGE_ALPHA)

In [ ]:
try:
    from catboost import CatBoostRegressor
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'catboost'])
    from catboost import CatBoostRegressor

X_cb, X_test_cb = X.copy(), X_test.copy()
for c in cat_cols:
    X_cb[c] = X_cb[c].astype(str).astype(object)
    X_test_cb[c] = X_test_cb[c].astype(str).astype(object)

X_hold_tr_cb = pd.concat([X_cb.iloc[hold_tr].reset_index(drop=True),
                           encode_author_features(AUTHOR_TRAIN[hold_tr], hold_stats, own_y=LOG_Y[hold_tr])], axis=1)
X_hold_va_cb = pd.concat([X_cb.iloc[hold_va].reset_index(drop=True),
                           encode_author_features(AUTHOR_TRAIN[hold_va], hold_stats, own_y=None)], axis=1)

CB_CFG = {}
for mode in ['log', 'growth']:
    y_mode = y_target(mode)
    cb = CatBoostRegressor(iterations=2000, learning_rate=0.05, depth=6, loss_function='RMSE',
                            random_seed=SEED, early_stopping_rounds=100, verbose=0)
    cb.fit(X_hold_tr_cb, y_mode[hold_tr], cat_features=cat_cols, eval_set=(X_hold_va_cb, y_mode[hold_va]), use_best_model=True)
    CB_CFG[mode] = dict(iters=int(cb.get_best_iteration()) + 1)
print(CB_CFG)

## 5. Cross-Validated Model Comparison

Every model is refit inside every fold, and the creator encoding is rebuilt fresh inside every
fold too (never computed once on the full data) — this is the exact discipline that fixed the
leakage/overfitting problem from the earlier attempt.

In [ ]:
def fit_predict_lgb(mode, X_fit, y_fit, X_pred, seed=SEED):
    cfg = LGB_CFG[mode]
    dtr = lgb.Dataset(X_fit, label=y_fit, categorical_feature=cat_cols)
    booster = lgb.train(lgb_params(cfg['lr'], cfg['objective'], seed), dtr, num_boost_round=cfg['iters'])
    return booster.predict(X_pred)

def fit_predict_rf(mode, X_fit, y_fit, X_pred, seed=SEED):
    rf = RandomForestRegressor(n_estimators=RF_TREES, max_features=0.33, min_samples_leaf=5, n_jobs=-1, random_state=seed)
    rf.fit(X_fit, y_fit)
    return rf.predict(X_pred)

def fit_predict_ridge(mode, X_fit, y_fit, X_pred, seed=SEED):
    pipe = Pipeline([('prep', make_ridge_prep()), ('model', Ridge(alpha=RIDGE_ALPHA))])
    pipe.fit(X_fit, y_fit)
    return pipe.predict(X_pred)

def fit_predict_cat(mode, X_fit, y_fit, X_pred, seed=SEED):
    cb = CatBoostRegressor(iterations=CB_CFG[mode]['iters'], learning_rate=0.05, depth=6,
                            loss_function='RMSE', random_seed=seed, verbose=0)
    cb.fit(X_fit, y_fit, cat_features=cat_cols)
    return cb.predict(X_pred)

# name -> (fit_predict fn, (train frame, test frame), list of target modes to run)
MODELS = {
    'lgb':   (fit_predict_lgb,   (X, X_test),       ['log', 'growth', 'tweedie']),
    'rf':    (fit_predict_rf,    (X_rf, X_test_rf), ['log']),
    'ridge': (fit_predict_ridge, (X, X_test),       ['log']),
    'cat':   (fit_predict_cat,   (X_cb, X_test_cb), ['log', 'growth']),
}

oof, oof_unclipped = {}, {}
for name, (fn, (Xm, _), modes) in MODELS.items():
    for mode in modes:
        key = f'{name}_{mode}'
        y_mode = y_target(mode)
        preds = np.zeros((N_SEEDS, len(Xm)))
        for s in range(N_SEEDS):
            for tr_i, va_i in tqdm(splits[s], desc=f'{key} | repeat {s + 1}/{N_SEEDS}', leave=False):
                stats = author_group_stats(tr_i)
                X_fit = pd.concat([Xm.iloc[tr_i].reset_index(drop=True),
                                    encode_author_features(AUTHOR_TRAIN[tr_i], stats, own_y=LOG_Y[tr_i])], axis=1)
                X_pred = pd.concat([Xm.iloc[va_i].reset_index(drop=True),
                                     encode_author_features(AUTHOR_TRAIN[va_i], stats, own_y=None)], axis=1)
                p = fn(mode, X_fit, y_mode[tr_i], X_pred)
                preds[s, va_i] = to_views(mode, p, base_train[va_i])
        oof_unclipped[key] = preds
        oof[key] = finalize(preds, last_train)
        print(f'{key:14s} done | raw RMSE = {np.mean([score(p)["raw"] for p in oof[key]]):,.0f}')

## 6. Ensembling: equal-weight vs. regularized stacking

Two blends are compared:
- **Equal-weight** (the v1 approach): safe, but leaves performance on the table.
- **Shrunk stacking**: fit unconstrained least-squares weights on the OOF predictions (in
  `log1p(views)` space, so no single viral video's raw magnitude dominates the fit), clip any
  negative weights to zero and renormalize, then blend that learned weight vector with the equal
  weight vector using a shrinkage factor `lambda` swept over a small grid. `lambda=0` reproduces
  the equal-weight blend exactly; `lambda=1` is the fully learned weights. The value of `lambda`
  is picked by the same CV raw RMSE used everywhere else in the notebook.

In [ ]:
def summarize(name, preds_by_repeat):
    df = pd.DataFrame([score(p) for p in preds_by_repeat])
    return {'model': name, 'raw_RMSE': df['raw'].mean(), 'raw_RMSE_sd': df['raw'].std(ddof=0),
            'log_RMSE': df['log'].mean(), 'raw_RMSE_without_top10': df['no_top10'].mean()}

BLEND_MEMBERS = list(oof.keys())

def equal_blend(members):
    return finalize(np.mean([oof[k] for k in members], axis=0), last_train)

def fit_stack_weights(members):
    """Unconstrained least-squares weights in log1p(views) space, clipped to >=0 and renormalized."""
    P = np.column_stack([np.log1p(np.mean(oof[k], axis=0)) for k in members])   # (n_rows, n_models)
    w, *_ = np.linalg.lstsq(P, y_log, rcond=None)
    w = np.clip(w, 0, None)
    return w / w.sum() if w.sum() > 0 else np.ones(len(members)) / len(members)

def shrunk_blend(members, lam):
    w_eq = np.ones(len(members)) / len(members)
    w_lstsq = fit_stack_weights(members)
    w = (1 - lam) * w_eq + lam * w_lstsq
    views_per_repeat = [np.average([oof[k][s] for k in members], axis=0, weights=w) for s in range(N_SEEDS)]
    return finalize(np.array(views_per_repeat), last_train)

rows = [summarize(k, v) for k, v in oof.items()]
rows.append(summarize('BLEND_equal', equal_blend(BLEND_MEMBERS)))
for lam in [0.25, 0.5, 0.75, 1.0]:
    rows.append(summarize(f'BLEND_shrunk_lambda{lam}', shrunk_blend(BLEND_MEMBERS, lam)))

results = pd.DataFrame(rows).sort_values('raw_RMSE').reset_index(drop=True)
display(results.style.format({'raw_RMSE': '{:,.0f}', 'raw_RMSE_sd': '{:,.0f}', 'log_RMSE': '{:.4f}',
                               'raw_RMSE_without_top10': '{:,.0f}'}))
print('Best row by raw RMSE:', results.iloc[0]['model'])

## 7. Error Analysis (best blend, OOF)

In [ ]:
best_model_name = results.iloc[0]['model']
if best_model_name == 'BLEND_equal':
    best_pred = equal_blend(BLEND_MEMBERS).mean(axis=0)
elif best_model_name.startswith('BLEND_shrunk'):
    lam = float(best_model_name.split('lambda')[1])
    best_pred = shrunk_blend(BLEND_MEMBERS, lam).mean(axis=0)
else:
    best_pred = oof[best_model_name].mean(axis=0)

err = pd.DataFrame({'video_id': train_feat['video_id'].values, 'topic': train_feat['topic'].values,
                     'actual': y_raw, 'predicted': best_pred})
err['residual'] = err['predicted'] - err['actual']
print(f"Overall bias (mean residual): {err['residual'].mean():,.0f}")
print(f"Share of videos over-predicted: {(err['residual'] > 0).mean():.1%}")
display(err.reindex(err['residual'].abs().sort_values(ascending=False).index).head(10)
        [['video_id', 'topic', 'actual', 'predicted', 'residual']])

## 8. Final Submission

The creator encoding for the final fit uses **all of train** as the stats source: training rows
still get the leave-one-out treatment (otherwise a single-video creator's feature would just equal
its own target), and test rows get the plain aggregate.

In [ ]:
final_stats = author_group_stats(np.arange(len(X)))
train_enc_final = encode_author_features(AUTHOR_TRAIN, final_stats, own_y=LOG_Y)
test_enc_final = encode_author_features(AUTHOR_TEST, final_stats, own_y=None)

FINAL_MEMBERS = BLEND_MEMBERS   # or restrict to results.head(...) members if you want a smaller ensemble
FINAL_LAMBDA = None
if best_model_name.startswith('BLEND_shrunk'):
    FINAL_LAMBDA = float(best_model_name.split('lambda')[1])
    FINAL_WEIGHTS = (1 - FINAL_LAMBDA) * np.ones(len(FINAL_MEMBERS)) / len(FINAL_MEMBERS) + FINAL_LAMBDA * fit_stack_weights(FINAL_MEMBERS)
else:
    FINAL_WEIGHTS = np.ones(len(FINAL_MEMBERS)) / len(FINAL_MEMBERS)

test_views = {}
for key in tqdm(FINAL_MEMBERS, desc='Training final models on the full training set'):
    name, mode = key.split('_', 1)
    fn, (Xm_train, Xm_test), _ = MODELS[name]
    X_fit_final = pd.concat([Xm_train.reset_index(drop=True), train_enc_final], axis=1)
    X_pred_final = pd.concat([Xm_test.reset_index(drop=True), test_enc_final], axis=1)
    p = fn(mode, X_fit_final, y_target(mode), X_pred_final)
    test_views[key] = to_views(mode, p, base_test)

final_views = finalize(np.average([test_views[k] for k in FINAL_MEMBERS], axis=0, weights=FINAL_WEIGHTS), last_test)

submission = pd.DataFrame({'video_id': test_feat['video_id'].values,
                            'target_day30_views': np.round(final_views).astype(np.int64)})
submission = sample_submission[['video_id']].merge(submission, on='video_id', how='left')
assert len(submission) == len(sample_submission)
assert submission['video_id'].equals(sample_submission['video_id'])
assert submission['target_day30_views'].notna().all()
submission['target_day30_views'] = submission['target_day30_views'].astype(np.int64)
assert (submission['target_day30_views'] >= 0).all()
submission.to_csv('submission.csv', index=False)
print('Submission shape:', submission.shape)
display(submission.head())

## 9. Summary

- **Feature engineering:** same engagement/creator-size/metadata features as v1, plus a
  leave-one-out, log-scale, smoothed creator historical-performance encoding computed fresh
  inside every CV fold and again for the final full-data fit.
- **Models:** LightGBM (log, growth, and Tweedie objectives), Random Forest, Ridge, CatBoost.
- **Ensembling:** equal-weight and shrinkage-regularized stacking are both evaluated by CV raw
  RMSE; whichever wins is used for the final submission.
- **Post-processing:** predictions are clipped to be non-negative and never below the last
  observed view count (validated as a large win in v1).
- **Open question for you to report back:** the per-key raw RMSE printout in section 5, and the
  results table in section 6 — those numbers will tell us whether the Tweedie model and/or the
  shrunk stacking actually beat v1's ~59K blended RMSE, and which piece is doing the work.